# So sánh 2 hướng: ResNet-18 pretrained vs CNN tự xây

Chạy sau khi đã train xong 2 notebook `Huong1_ResNet18_pretrained.ipynb` và `Huong2_CNN_tu_xay.ipynb`.
Notebook đánh giá cả hai model trên **cùng tập test**, vẽ biểu đồ và lưu hình vào `ket_qua/hinh_slide/` để đưa vào slide.

Trước khi chạy, upload lên Google Drive thư mục `MyDrive/bienbao/`:
- `cnn_dataset_32.zip` (nén từ `Data_Processed/cnn_dataset_32`, tạo bằng `Data_Pipeline/build_cnn_crops.py`)

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từng cell.
Model và lịch sử train được lưu vào `MyDrive/bienbao/ket_qua/`.

In [1]:
import os

try:  # Chạy trên Colab: lấy dataset từ Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/bienbao'
    if not os.path.exists('/content/cnn_dataset_32'):
        !unzip -q {DRIVE_DIR}/cnn_dataset_32.zip -d /content/
    DATA_DIR = '/content/cnn_dataset_32/'
    OUT_DIR = DRIVE_DIR + '/ket_qua/'
except ImportError:  # Chạy trên máy
    DATA_DIR = '../../Data_Processed/cnn_dataset_32/'
    OUT_DIR = '../Inference/'
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
import glob
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.io as tv_io
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIG_DIR = OUT_DIR + 'hinh_slide/'
os.makedirs(FIG_DIR, exist_ok=True)

In [3]:
IMG_HEIGHT = 32
IMG_WIDTH = 32
IMG_CHS = 3

# Tên thư mục = nhãn (lớp IE đã bỏ vì quá ít ảnh)
DATA_LABELS = ['0_P', '1_W', '2_R', '3_I', '5_S']
N_CLASSES = len(DATA_LABELS)

class MyDataset(Dataset):
    def __init__(self, data_dir):
        imgs = []
        labels = []
        for l_idx, label in enumerate(DATA_LABELS):
            data_paths = sorted(glob.glob(data_dir + label + '/*.png'))
            for path in data_paths:
                imgs.append(tv_io.read_image(path, tv_io.ImageReadMode.RGB))  # 3 x 32 x 32
                labels.append(l_idx)
        self.xs = (torch.stack(imgs).float() / 255).to(device)  # Normalize values from 0 to 1
        self.ys = torch.tensor(labels).to(device)

    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx]

    def __len__(self):
        return len(self.xs)


test_data = MyDataset(DATA_DIR + 'test/')
test_loader = DataLoader(test_data, batch_size=64)
CLASS_NAMES = ['P (cấm)', 'W (nguy hiểm)', 'R (hiệu lệnh)', 'I (chỉ dẫn)', 'S (phụ)']
y_true = test_data.ys.cpu().numpy()
print('Số ảnh test:', len(test_data))

Số ảnh test: 2259


## Load 2 model và lịch sử train

In [4]:
MODELS = {
    'ResNet-18 pretrained': 'resnet18_pretrained',
    'CNN tự xây': 'custom_cnn',
}
models, histories = {}, {}
for title, name in MODELS.items():
    models[title] = torch.load(OUT_DIR + name + '.pth', weights_only=False, map_location=device).eval()
    histories[title] = json.load(open(OUT_DIR + name + '_history.json'))

## Dự đoán trên tập test

In [5]:
def predict(model):
    preds = []
    with torch.no_grad():
        for x, y in test_loader:
            preds.append(model(x).argmax(dim=1).cpu())
    return torch.cat(preds).numpy()


def inference_time_ms(model, n=200):
    # Thời gian dự đoán 1 ảnh (batch = 1), trung bình n lần
    x = test_data.xs[:1]
    with torch.no_grad():
        for _ in range(10):  # chạy nháp cho GPU "nóng" lên
            model(x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(n):
            model(x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    return (time.time() - start) / n * 1000


preds, rows = {}, []
for title, model in models.items():
    preds[title] = predict(model)
    rows.append({
        'Model': title,
        'Số tham số': sum(p.numel() for p in model.parameters()),
        'Dung lượng (MB)': os.path.getsize(OUT_DIR + MODELS[title] + '.pth') / 1e6,
        'Thời gian train (phút)': histories[title]['train_time_s'] / 60,
        'Thời gian dự đoán (ms/ảnh)': inference_time_ms(model),
        'Test accuracy (%)': (preds[title] == y_true).mean() * 100,
        'Macro F1 (%)': f1_score(y_true, preds[title], average='macro') * 100,
    })

## Bảng so sánh (copy vào slide)

In [6]:
keys = list(rows[0].keys())
print('| ' + ' | '.join(keys) + ' |')
print('|' + '---|' * len(keys))
for r in rows:
    cells = [r['Model'], f"{r['Số tham số']:,}"] + [f'{r[k]:.2f}' for k in keys[2:]]
    print('| ' + ' | '.join(cells) + ' |')

| Model | Số tham số | Dung lượng (MB) | Thời gian train (phút) | Thời gian dự đoán (ms/ảnh) | Test accuracy (%) | Macro F1 (%) |
|---|---|---|---|---|---|---|
| ResNet-18 pretrained | 11,309,125 | 45.33 | 11.66 | 3.11 | 99.47 | 97.61 |
| CNN tự xây | 1,149,189 | 4.61 | 2.92 | 0.49 | 99.47 | 98.77 |


## Biểu đồ quá trình train

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for title, h in histories.items():
    axes[0].plot(h['valid_acc'], marker='o', label=title)
    axes[1].plot(h['valid_loss'], marker='o', label=title)
axes[0].set_title('Valid accuracy theo epoch'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Valid loss theo epoch'); axes[1].set_xlabel('Epoch')
for ax in axes:
    ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR + 'qua_trinh_train.png', dpi=150)
plt.show()

/tmp/ipykernel_11540/1028687142.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Biểu đồ cột: accuracy, F1, số tham số, tốc độ

In [8]:
titles = [r['Model'] for r in rows]
colors = ['tab:blue', 'tab:orange']
metrics = [('Test accuracy (%)', 'Test accuracy (%)'), ('Macro F1 (%)', 'Macro F1 (%)'),
           ('Số tham số', 'Số tham số (triệu)'), ('Thời gian dự đoán (ms/ảnh)', 'Thời gian dự đoán (ms/ảnh)')]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (key, label) in zip(axes, metrics):
    values = [r[key] / 1e6 if key == 'Số tham số' else r[key] for r in rows]
    bars = ax.bar(titles, values, color=colors)
    ax.bar_label(bars, fmt='%.2f')
    ax.set_title(label)
    if '%' in key:
        ax.set_ylim(min(values) - 5, 100)
plt.tight_layout()
plt.savefig(FIG_DIR + 'so_sanh_chi_so.png', dpi=150)
plt.show()

/tmp/ipykernel_11540/3236788997.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ma trận nhầm lẫn (confusion matrix)

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, title in zip(axes, models):
    cm = confusion_matrix(y_true, preds[title])
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=30)
    ax.set_title(title)
plt.tight_layout()
plt.savefig(FIG_DIR + 'confusion_matrix.png', dpi=150)
plt.show()

/tmp/ipykernel_11540/3932914190.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Precision / Recall / F1 từng lớp

In [10]:
for title in models:
    print('=' * 20, title, '=' * 20)
    print(classification_report(y_true, preds[title], target_names=CLASS_NAMES, digits=4))

==================== ResNet-18 pretrained ====================
               precision    recall  f1-score   support

      P (cấm)     1.0000    1.0000    1.0000      1115
W (nguy hiểm)     0.9975    0.9901    0.9938       810
R (hiệu lệnh)     1.0000    0.9899    0.9949       199
  I (chỉ dẫn)     0.9691    0.9792    0.9741        96
      S (phụ)     0.8478    1.0000    0.9176        39

     accuracy                         0.9947      2259
    macro avg     0.9629    0.9918    0.9761      2259
 weighted avg     0.9952    0.9947    0.9948      2259

==================== CNN tự xây ====================
               precision    recall  f1-score   support

      P (cấm)     0.9946    0.9991    0.9969      1115
W (nguy hiểm)     1.0000    0.9963    0.9981       810
R (hiệu lệnh)     0.9847    0.9698    0.9772       199
  I (chỉ dẫn)     0.9792    0.9792    0.9792        96
      S (phụ)     0.9750    1.0000    0.9873        39

     accuracy                         0.9947      2259

## Một số ảnh bị đoán sai

In [11]:
for title in models:
    wrong = np.where(preds[title] != y_true)[0][:10]
    if len(wrong) == 0:
        continue
    fig, axes = plt.subplots(1, len(wrong), figsize=(2 * len(wrong), 2.6), squeeze=False)
    for ax, i in zip(axes[0], wrong):
        ax.imshow(test_data.xs[i].permute(1, 2, 0).cpu())
        ax.set_title(f"Đúng: {DATA_LABELS[y_true[i]][2:]}\nĐoán: {DATA_LABELS[preds[title][i]][2:]}", fontsize=9)
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(FIG_DIR + f'anh_sai_{MODELS[title]}.png', dpi=150)
    plt.show()

/tmp/ipykernel_11540/907448563.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Tải thư mục `MyDrive/bienbao/ket_qua/` về máy: 2 file `.pth` chép vào `Inference/`, các hình trong `hinh_slide/` dùng cho slide.